In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,f1_score
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder,StandardScaler
import torch
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import Dataset,DataLoader
from helper import fn_plot_torch_hist ,fn_plot_confusion_matrix

In [3]:
RANDOM_STATE = 24  # Remember to remove at the time of promotion to production(RANDOM_STATE = None) 
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(seed = RANDOM_STATE)

NOISE = 0.2
N_EPOCHS = 100 # Number of Epochs
ALPHA = .001  # Learning Rate 
WEIGHT_DECAY=1e-4
BATCH_SIZE=32
N_SAMPLES = 1000  # number of sample to create
TEST_SIZE = 0.2
TRAIN_SIZE=int(BATCH_SIZE*18)
PATIENCE=20
LR_PATIENCE=5
LR_FACTOR=0.1

params = {
    'legend.fontsize':'medium',
    'figure.figsize':(15,6),
    'axes.labelsize':'medium',
    'axes.titlesize':'medium',
    'xtick.labelsize':'medium',
    'ytick.labelsize':'medium'
}
plt.rcParams.update(params)
CMAP = plt.cm.coolwarm
plt.style.use('seaborn-v0_8-darkgrid') 

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'}")

Using cpu


In [5]:
data_df = pd.read_csv("F:\DEEP NEURAL NETWORK\DAY6(kanak)\inosphere.csv",header=None)
data_df

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
0,1,0,0.99539,-0.05889,0.85243,0.02306,0.83398,-0.37708,1.00000,0.03760,...,-0.51171,0.41078,-0.46168,0.21266,-0.34090,0.42267,-0.54487,0.18641,-0.45300,g
1,1,0,1.00000,-0.18829,0.93035,-0.36156,-0.10868,-0.93597,1.00000,-0.04549,...,-0.26569,-0.20468,-0.18401,-0.19040,-0.11593,-0.16626,-0.06288,-0.13738,-0.02447,b
2,1,0,1.00000,-0.03365,1.00000,0.00485,1.00000,-0.12062,0.88965,0.01198,...,-0.40220,0.58984,-0.22145,0.43100,-0.17365,0.60436,-0.24180,0.56045,-0.38238,g
3,1,0,1.00000,-0.45161,1.00000,1.00000,0.71216,-1.00000,0.00000,0.00000,...,0.90695,0.51613,1.00000,1.00000,-0.20099,0.25682,1.00000,-0.32382,1.00000,b
4,1,0,1.00000,-0.02401,0.94140,0.06531,0.92106,-0.23255,0.77152,-0.16399,...,-0.65158,0.13290,-0.53206,0.02431,-0.62197,-0.05707,-0.59573,-0.04608,-0.65697,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,1,0,0.83508,0.08298,0.73739,-0.14706,0.84349,-0.05567,0.90441,-0.04622,...,-0.04202,0.83479,0.00123,1.00000,0.12815,0.86660,-0.10714,0.90546,-0.04307,g
347,1,0,0.95113,0.00419,0.95183,-0.02723,0.93438,-0.01920,0.94590,0.01606,...,0.01361,0.93522,0.04925,0.93159,0.08168,0.94066,-0.00035,0.91483,0.04712,g
348,1,0,0.94701,-0.00034,0.93207,-0.03227,0.95177,-0.03431,0.95584,0.02446,...,0.03193,0.92489,0.02542,0.92120,0.02242,0.92459,0.00442,0.92697,-0.00577,g
349,1,0,0.90608,-0.01657,0.98122,-0.01989,0.95691,-0.03646,0.85746,0.00110,...,-0.02099,0.89147,-0.07760,0.82983,-0.17238,0.96022,-0.03757,0.87403,-0.16243,g


In [6]:
data_df.isna().sum().sum()

np.int64(0)

In [7]:
labels=data_df[data_df.columns[-1]]
features_df=data_df.drop(data_df.columns[-1],axis=1)
features_df.shape,labels.shape

((351, 34), (351,))

In [8]:
X_train,X_test,y_train,y_test=train_test_split(features_df,
                                               labels,
                                               test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE,
                                               stratify=labels)

In [9]:
class DataTransformer:
    def __init__ (self):
        self.scaler=StandardScaler()
        self.encoder=LabelEncoder()
        self.is_fitted=False
        
    def fit(self,X,y):
        self.scaler.fit(X)
        self.encoder.fit(y)
        self.is_fitted=True
        return self
        
    def transform(self,X,y):
        if not self.fitted:
            raise ValueError('Transformer must be fitted first')
        return self.scaler.transform(X),self.encoder.transform(y)
        
    def fit_tranform(self,X,y):
        self.fit(X,y)
        return self.transform(X,y)

In [ ]:
class IonoDS(Dataset):
    def __init__(self,
                dataframe:pd.DataFrame,
                device:str=device,
                is_train=True,
                label_col=None,
                transformer=None):
        
        super(IonoDS, self).__init__()
        self.df = dataframe
        self.device = device
        self.is_train = is_train
        self.label_col = label_col
        self.transformer = transformer
        y = self.df[label_col].to_numpy()
        X = self.df.drop(label_col, axis=1)

        if self.is_train:
            self.transformer=DataTransformer()
            self.features,self.labels=self.transformer.fit_tranform(X,y)
      
        else:
            if self.transformer is None:
                raise ValueError('Test set requires pre-fitted transformer')
            self.features, self.labels = self.transformer.transform(X, y)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        feature = self.features[idx]
        label = self.labels[idx]
        
        feature = torch.tensor(feature, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.int64)
        return feature, label
        
    

In [ ]:
label_col = 34
train_dataset = IonoDS(train_df, is_train=True, label_col=label_col)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


In [ ]:
label_col = 34
test_dataset = IonoDS(test_df, is_train=False, label_col=label_col,transformer=)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [11]:
class Model(nn.Module):
    def __init__(self,input_dim):
        super(Model,self.__init__())
        dor1=.5
        dor2=.15
        dor3=.25
        self.layer1 = nn.Linear(input_dim, 26)
        self.bn1 = nn.BatchNorm1d(26)
        self.activ1 = nn.ReLU()
        self.do1 = nn.Dropout(dor1)
        
        self.layer2 = nn.Linear(26, 18)
        self.bn2 = nn.BatchNorm1d(18)
        self.activ2 = nn.ReLU()
        self.do2 = nn.Dropout(dor2)
        
        self.layer3 = nn.Linear(18, 10)
        self.bn3 = nn.BatchNorm1d(10)
        self.activ3 = nn.ReLU()
        self.do3 = nn.Dropout(dor3)
        
        self.layer4 = nn.Linear(10, 2)
    
    def forward(self, x):
        x = self.do1(self.activ1(self.bn1(self.layer1(x))))
        x = self.do2(self.activ2(self.bn2(self.layer2(x))))
        x = self.do3(self.activ3(self.bn3(self.layer3(x))))
        output=self.layer4(x)
        return output

In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),  # Parameters to optimize
                             lr=ALPHA,            # learning rate
                             weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode='min', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=1e-6
)

train_losses, test_losses = [], []
train_acc, test_acc = [], []
n_epoch = []

minLoss = float('inf')
counter = 0
savePath = 'F:\DEEP NEURAL NETWORK\DAY7(in_pytorch)'


NameError: name 'model' is not defined

In [ ]:
train_losses, test_losses = [],[]
train_accuracy, test_accuracy = [],[]
n_epoch = []

minLoss = float('inf')
counter = 0

for epoch in range(EPOCHS):
    model.train
    train_loss = 0
    test_loss = 0
    train_acc= 0
    test_acc = 0
    
    for batch_idx, (train_X, train_y) in enumerate(train_loader):
        train_X, train_y = train_X.to(device), train_y.to(device)
        predict_prob = model(train_X) # Forward propagation
        batch_loss = loss_fn(predict_prob, train_y) # calculate loss
        train_loss += (batch_loss - train_loss) / (batch_idx+1)
        
        # Back propagation
        optimizer.zero_grad() # reset
        batch_loss.backward()       # back Prop (dWs dbs)
        optimizer.step() # W = W + alpha * dW
        
        _, y_pred = torch.max(predict_prob, 1)
        batch_acc = accuracy_score(train_y.cpu().numpy(), y_pred.data.cpu())
        train_acc += (batch_acc - train_acc) / (batch_idx+1)
    
    train_losses.append(train_loss.item())
    train_acces.append(train_acc)
    
    model.eval()
    with torch.inference_mode():
        for batch_idx, (test_X, test_y) in enumerate(test_loader):
            test_X, test_y = test_X.to(device), test_y.to(device)
            predict_prob_tst = model(test_X) # Forward propagation
            tbatch_loss = loss_fn(predict_prob_tst, test_y) # calculate loss
            test_loss += (tbatch_loss - test_loss) / (batch_idx+1)
            
            _, y_pred = torch.max(predict_prob_tst, 1)
            tbatch_acc = accuracy_score(test_y.cpu().numpy(), y_pred.data.cpu())
            test_acc += (tbatch_acc - test_acc) / (batch_idx+1)
    
    test_losses.append(test_loss.item())
    test_acces.append(test_acc)
    
    n_epoch.append(epoch)
    
    scheduler.step(test_loss)
    ### Add for early stopping
    if test_loss < minLoss:
        minLoss = test_loss
        counter = 0
        torch.save({'epoch': epoch+1'
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': test_loss }, savePath)
    else:
        counter+=1
    if counter> PATIENCE:
        break
        
    if epoch % 10 == 0:
        print(f'At epoch - {epoch:4d}: Train Loss - {train_loss.item():.4f}'
              f' Test Loss - {test_loss.item():.4f}'
              f' Train Acc - {train_acc:.4f}'
              f' Test Loss - {test_acc:.4f}')


In [ ]:
y_train, y_pred = [], []
model.eval()
with torch.inference_mode():
    for batch_idx, (train_X, train_y) in enumerate(train_loader):
        train_X, train_y = train_X.to(device), train_y.to(device)
        predict_prob_tst = model(train_X) # Forward propagation
        preds = torch.argmax(predict_prob_tst, dim=1)
        y_pred.extend(preds.cpu().numpy())
        y_train.extend(train_y.cpu().numpy())

print(f'Train Acc: {accuracy_score(y_train, y_pred):0.5f}')


In [ ]:
y_test, y_pred = [],[]
model.eval()
with torch.inference_mode():
    for batch_idx, (test_X, test_y) in enumerate (test_loader):
        test_X, test_y = test_X.to(device), test_y.to(device)
        predict_prob_tst = model(test_X)
        preds = torch.argmax(predict_prob_tst, dim = 1)
        y_pred .extend(preds.cpu().numpy())
        y_test.extend(test_y.cpu().numpy())

print(f'Test Acc: {accuracy_score(y_test, y_pred):0.5f})
print(classification_report(y_test,y_pred))